In [ ]:
print(spark.version)

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, concat, lit, initcap, hour, round, floor, unix_timestamp, dayofweek, count, avg, max, min, sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType,TimestampType

#schema for yellow files
yellow_taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])
#schema for green files
green_taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("lpep_pickup_datetime", TimestampType(), True),
    StructField("lpep_dropoff_datetime", TimestampType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),	
    StructField("passenger_count", LongType(), True),	
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("ehail_fee", IntegerType(), True),	
    StructField("improvement_surcharge", DoubleType(), True),	
	StructField("total_amount", DoubleType(), True),
    StructField("payment_type", DoubleType(), True),
    StructField("trip_type", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
])

# --- 1. Load Data ---
S3_BUCKET_NAME = "robot-dreams-source-data"
YELLOW_TAXI_PATH = f"s3a://{S3_BUCKET_NAME}/home-work-1-unified/nyc_taxi/yellow/"
GREEN_TAXI_PATH = f"s3a://{S3_BUCKET_NAME}/home-work-1-unified/nyc_taxi/green/"

#read yellow
#filter on read stage = small but optimization
print(f"Loading Yellow Taxi data from: {YELLOW_TAXI_PATH}")
yellow_taxi_df = spark.read \
                     .option("mergeSchema", "true") \
                     .option("recursiveFileLookup", "true") \
                     .schema(yellow_taxi_schema).parquet(YELLOW_TAXI_PATH) \
                     .withColumn("taxi_type", lit('yellow')) \
                     .filter(
                        (col("trip_distance") > 0.1) &
                        (col("fare_amount") > 2)
                    )

#yellow_unified = yellow_taxi_df.select(col("VendorID").cast("long"))

#read green
#filter on read stage = small but optimization
#filter anomalies
print(f"Loading Green Taxi data from: {GREEN_TAXI_PATH}")
green_taxi_df = spark.read \
                     .option("mergeSchema", "true") \
                     .option("recursiveFileLookup", "true") \
                     .schema(green_taxi_schema).parquet(GREEN_TAXI_PATH) \
                     .withColumn("taxi_type", lit('green')) \
                     .filter(
                        (col("trip_distance") > 0.1) &
                        (col("fare_amount") > 2)
                    )
#green_unified = green_taxi_df.select(col("VendorID").cast("long"))

print(f"Yellow Taxi data loaded. Number of records: {yellow_taxi_df.count()}")
print(f"Green Taxi data loaded. Number of records: {green_taxi_df.count()}")
print("Schema:")
green_taxi_df.printSchema()

#unoin dataframes green + yellow
raw_trips_df = yellow_taxi_df.unionByName(green_taxi_df, allowMissingColumns=True)

#raw_trips_df.take(1)

In [ ]:
#add new columns: extract new info from existing columns
#duration for another anomalies filter

add_duration = raw_trips_df.withColumn("duration_min", (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60)\
    .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", dayofweek("tpep_pickup_datetime"))

#another anomalies filter
filter_duration = add_duration.filter("duration_min > 1")

print(f"After UNION and filters: {filter_duration.count()}")

In [ ]:
lookup_path = f"s3://robot-dreams-source-data/home-work-1-unified/nyc_taxi/taxi_zone_lookup.csv"

#read csv, no comments
zones_df = spark.read.option("header", True).csv(lookup_path) \
    .select(
        col("LocationID").cast("int").alias("location_id"),
        col("Zone").alias("zone")
    )

#we need to join twice, so create two dfs to awoid name conflict on join stage
zones_pickup = zones_df.alias("pickup_zones")
zones_dropoff = zones_df.alias("dropoff_zones")

#join dictionary and add informative columns
joined_df = filter_duration \
    .join(zones_pickup, col("PULocationID") == col("pickup_zones.location_id"), "left") \
    .join(zones_dropoff, col("DOLocationID") == col("dropoff_zones.location_id"), "left") \
    .withColumn("pickup_zone", when(col("pickup_zones.zone").isNull(), "Unknown").otherwise(col("pickup_zones.zone"))) \
    .withColumn("dropoff_zone", when(col("dropoff_zones.zone").isNull(), "Unknown").otherwise(col("dropoff_zones.zone")))

In [30]:
#use aggregation and conditional expression "case when..."
zone_summary = joined_df.groupBy("pickup_zone").agg(
    count("*").alias("total_trips"),
    round(avg("trip_distance"), 2).alias("avg_trip_distance"),
    round(avg("total_amount"), 2).alias("avg_total_amount"),
    round(avg("tip_amount"), 2).alias("avg_tip_amount"),
    max("trip_distance").alias("max_trip_distance"),
    min("tip_amount").alias("min_tip_amount"),
    sum(when(col("taxi_type") == "yellow", 1).otherwise(0)).alias("yellow_count"),
    sum(when(col("taxi_type") == "green", 1).otherwise(0)).alias("green_count")
)

#save file on s3, repartition 1 = one file
zone_summary.repartition(1).write.mode('overwrite').parquet("s3://vyaskov-emr-studio/results/zone_statistic/19072025/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [31]:
#calculate share and quantity by days and zones
zone_day_stats = joined_df.groupBy("pickup_zone", "pickup_day_of_week").agg(
    count("*").alias("total_trips"),
    round(sum(when(col("total_amount") > 30, 1).otherwise(0)) / count("*"), 4).alias("high_fare_share")
)

#save file on s3, repartition 1 = one file
zone_day_stats.repartition(1).write.mode('overwrite').parquet("s3://vyaskov-emr-studio/results/zone_days_statstic/19072025/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…